<a href="https://colab.research.google.com/github/gigantee04/ISA444_Spring2026.github.io/blob/main/FoundationModelNotebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
##!pip install timecopilot utilsforecast

import pandas as pd
from timecopilot import TimeCopilotForecaster
from timecopilot.models.foundation.chronos import Chronos
from timecopilot.models.foundation.moirai import Moirai
from timecopilot.models.foundation.timesfm import TimesFM

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mape, rmse, mae, bias

In [ ]:
df = (
    pd.read_parquet("/content/sample_hotels-1.parquet")
    .query("unique_id not in ['hotel_77', 'hotel_28']")
   .assign(
        ds = lambda x: pd.to_datetime(x["ds"]),
        unique_id = lambda x: x["unique_id"].astype(str),
        y = lambda x: x["y"].astype(float),
   )
    [['unique_id', 'ds', 'y']]
)


TimeCopilotForecaster.plot(df, engine='plotly')

In [ ]:
tcf = TimeCopilotForecaster(
    models = [
        Chronos(repo_id="amazon/chronos-bolt-small"),
        Moirai(),
        TimesFM(repo_id="google/timesfm-2.5-200m-pytorch", alias="TimesFM-2.5"),
    ]
)

t_cv = tcf.cross_validation(
    df = df,
    h = 28, freq = 'D',
    n_windows = 12,
    step_size = 28
)

In [ ]:
eval_transformers = evaluate(
    df = t_cv,
    metrics = [mape, rmse, mae, bias],
    models = ['Chronos', 'Moirai', 'TimesFM-2.5']
)

eval_transformers = eval_transformers.drop(columns=['cutoff']).groupby(['unique_id','metric']).mean().reset_index(inplace=False)
eval_transformers

In [ ]:
# 1. Aggregate to get one row per model per metric (Average across all IDs)
overall_metrics = eval_transformers.groupby('metric').mean(numeric_only=True)

# 2. Identify the "Winner" for each series and each metric
# We find the column name (model) with the minimum value for each row
model_columns = ['Chronos', 'Moirai', 'TimesFM-2.5']
eval_transformers['winner'] = eval_transformers[model_columns].idxmin(axis=1)

# 3. Count how many times each model won
model_wins = eval_transformers.groupby(['metric', 'winner']).size().unstack(fill_value=0)

print("--- Overall Average Metrics ---")
print(overall_metrics)
print("\n--- Model Win Counts ---")
print(model_wins)

In [ ]:
# Generate forecasts for the next 28 days
forecasts_df = tcf.forecast(df=df, h=28, freq='D')

# Export to CSV
forecasts_df.to_csv("final_forecasts.csv", index=False)
print("Forecasts exported to final_forecasts.csv")

In [ ]:
recent_history = df.groupby('unique_id').tail(60)

TimeCopilotForecaster.plot(
    df=recent_history,
    forecasts_df=forecasts_df,
    models=['Chronos', 'Moirai', 'TimesFM-2.5'],
    engine='plotly',
    max_ids=5
)